# Verify Preprocessed Keypoint Data

This notebook loads preprocessed keypoint data and verifies:
1. Data structure and dimensions
2. Keypoint ordering matches XML sites
3. Scale matches MuJoCo model
4. Visualization with colored sites overlaid on model

In [1]:

import sys
from pathlib import Path

# Add project root to Python path FIRST to ensure our modules take priority
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"Added {project_root} to Python path")
elif sys.path.index(str(project_root)) != 0:
    # Move to front if it exists but isn't first
    sys.path.remove(str(project_root))
    sys.path.insert(0, str(project_root))
    print(f"Moved {project_root} to front of Python path")

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ["XLA_FLAGS"] = "--xla_gpu_triton_gemm_any=True"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Use GPU 0

# JAX setup
import jax
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
# Note: jax_persistent_cache_enable_xla_caches may not be available in all JAX versions
try:
    jax.config.update("jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir")
except AttributeError:
    pass  # Skip if not available in this JAX version
jax.config.update("jax_default_matmul_precision", "high")

# Matplotlib setups
import matplotlib as mpl
mpl.rcParams.update({
    'font.size': 10,
    'axes.linewidth': 2,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.major.width': 2,
    'ytick.major.width': 2,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'pdf.fonttype': 42,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.facecolor': 'white',
    'pdf.use14corefonts': True,
    'svg.fonttype': 'none',
    'font.family': 'sans-serif',
    'font.serif': 'Arial',
})

Added /home/eabe/Research/MyRepos/3d_tracking_dataset to Python path


In [2]:
%load_ext autoreload
%autoreload 2

# Imports
import numpy as np
import jax.numpy as jnp
import mujoco
import matplotlib.pyplot as plt
import mediapy as media

from pathlib import Path
from tqdm.auto import tqdm
import utils.io_dict_to_hdf5 as ioh5
from utils.add_aligned_keypoint_sites import set_aligned_site_colors
from utils.stac_data_utils import (
    print_bout_dict_structure,
    
)
print("✓ Imports successful")

✓ Imports successful


## 1. Configuration

In [3]:
# Paths
project_root = Path('/home/eabe/Research/MyRepos/3d_tracking_dataset')
data_root = Path('/data2/users/eabe/datasets/Johnson_lab/free_walking/Predictions_3D_20260114-145343')

# Preprocessed data file
preprocessed_path = data_root / 'preprocessed_bout.h5'
stac_path = data_root / 'ik_output.h5'
# Model paths
skeleton_path = project_root / 'data' / 'fly50.json'
flybody_path = project_root /'assets/fruitfly_v1/fruitfly_v1_free.xml'
floor_path = project_root / 'assets/fruitfly_v1/floor.xml'
print(f"Preprocessed data: {preprocessed_path}")
print(f"Model: {flybody_path}")
print(f"Data exists: {preprocessed_path.exists()}")
print(f"Model exists: {flybody_path.exists()}")


Preprocessed data: /data2/users/eabe/datasets/Johnson_lab/free_walking/Predictions_3D_20260114-145343/preprocessed_bout.h5
Model: /home/eabe/Research/MyRepos/3d_tracking_dataset/assets/fruitfly_v1/fruitfly_v1_free.xml
Data exists: True
Model exists: True


## 2. Load Preprocessed Data

In [14]:
# Load HDF5 data
print("Loading preprocessed data...")
data_dict = ioh5.load(preprocessed_path, enable_jax=True)

print(f"\n✓ Loaded {len(data_dict)} bouts")
print(f"Bout keys: {list(data_dict.keys())[:5]}...")

# Examine first bout
bout_n = 113
bout_key = f'bout_{bout_n:03d}'
bout = data_dict[bout_key]

print(f"\nBout '{bout_key}' structure:")
for key, val in bout.items():
    if hasattr(val, 'shape'):
        print(f"  {key:20s}: shape {val.shape}, dtype {val.dtype}")
    elif isinstance(val, dict):
        print(f"  {key:20s}: dict with keys {list(val.keys())}")
    elif isinstance(val, list):
        print(f"  {key:20s}: list of {len(val)} items")
    else:
        print(f"  {key:20s}: {type(val).__name__}")

Loading preprocessed data...

✓ Loaded 130 bouts
Bout keys: ['bout_001', 'bout_002', 'bout_003', 'bout_004', 'bout_005']...

Bout 'bout_113' structure:
  alignment_info      : dict with keys ['rotation', 'scales', 'translation']
  keypoints           : shape (607, 50, 3), dtype float32
  kp_names            : list of 50 items
  orig_keypoints      : shape (607, 50, 3), dtype float32
  skeleton_edges      : shape (44, 2), dtype int32


In [15]:
# Align MuJoCo model's T1R coxa to keypoint T1R coxa (FIXED)
from utils.add_aligned_keypoint_sites import (
    add_aligned_keypoint_sites_to_model, 
    get_aligned_site_indices,
    set_aligned_site_colors,
    add_aligned_mocap_bodies,
    get_aligned_mocap_indices
)

# Step 1: Find T1R_ThxCx index in keypoint data
# bout_n = 2
kp_data = data_dict[f'bout_{bout_n:03d}']['keypoints']  # Shape: (T, N, 3)
kp_node_names = data_dict[f'bout_{bout_n:03d}']['kp_names']

print(f"Keypoint data shape: {kp_data.shape}")
print(f"Available keypoints: {kp_node_names[:10]}...")

# Find T1R_ThxCx index
try:
    t1r_coxa_idx = kp_node_names.index('T1R_ThxCx')
    print(f"\n✓ Found T1R_ThxCx at index {t1r_coxa_idx}")
except ValueError:
    print("\n⚠ T1R_ThxCx not found in keypoint names!")
    print("Available keypoints with 'T1R':")
    for i, name in enumerate(kp_node_names):
        if 'T1R' in name:
            print(f"  [{i}] {name}")
    t1r_indices = [i for i, name in enumerate(kp_node_names) if 'T1R' in name]
    if t1r_indices:
        t1r_coxa_idx = t1r_indices[0]
        print(f"\nUsing {kp_node_names[t1r_coxa_idx]} at index {t1r_coxa_idx} as reference")
    else:
        raise ValueError("No T1R keypoints found!")

# Step 2: Load models and add mocap bodies
spec = mujoco.MjSpec().from_file(flybody_path.as_posix())

# Load floor and attach fly WITH floor offset
suffix = '_fly'
floor_offset = -0.125  # Floor z-position
floor_spec = mujoco.MjSpec().from_file(floor_path.as_posix())
spawn_frame = floor_spec.worldbody.add_frame(
    pos=[0, 0, floor_offset],
    quat=[1, 0, 0, 0],
)
spawn_body = spawn_frame.attach_body(spec.body("thorax"), "", suffix=suffix)

# Add mocap bodies for keypoint visualization
floor_spec = add_aligned_mocap_bodies(
    floor_spec, 
    kp_node_names, 
    color_coded=True, 
    prefix='aligned_'
)

# Compile model
mj_model = floor_spec.compile()
print(f"\n✓ Model compiled - Total mocap bodies: {mj_model.nmocap}")

# Get mocap indices for keypoint visualization
mocap_indices = get_aligned_mocap_indices(mj_model, kp_node_names, prefix='aligned_')
print(f"✓ Mapped {len(mocap_indices)} mocap bodies")

# Step 3: Get T1R coxa site ID from MuJoCo model
t1r_coxa_site_name = f'tracking[T1R_ThxCx]{suffix}'
try:
    t1r_coxa_site_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_SITE, t1r_coxa_site_name)
    print(f"✓ Found MuJoCo T1R coxa site: {t1r_coxa_site_name} (ID: {t1r_coxa_site_id})")
except:
    print(f"⚠ Could not find site: {t1r_coxa_site_name}")
    print(f"Available tracking sites:")
    for i in range(mj_model.nsite):
        site_name = mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_SITE, i)
        if 'T1R' in site_name and 'tracking' in site_name:
            print(f"  {site_name}")
    raise ValueError("T1R coxa site not found in model")

# Initialize data
mj_data = mujoco.MjData(mj_model)
mujoco.mj_forward(mj_model, mj_data)

# Get T1R coxa position in model's reference pose
# This includes the floor offset already
ref_t1r_coxa_world = mj_data.site_xpos[t1r_coxa_site_id].copy()
print(f"\nT1R coxa in model reference (with floor offset): {ref_t1r_coxa_world}")
print(f"Floor offset: {floor_offset}")

# Get the root body position in reference pose
root_body_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, f'thorax{suffix}')
ref_root_pos = mj_data.xpos[root_body_id].copy()
print(f"Root body position in reference: {ref_root_pos}")

# Compute T1R coxa offset relative to root (in body frame)
t1r_coxa_offset_from_root = ref_t1r_coxa_world - ref_root_pos
print(f"T1R coxa offset from root: {t1r_coxa_offset_from_root}")

# Step 4: Render with aligned T1R coxa
scene_option = mujoco.MjvOption()
scene_option.sitegroup[:] = [1, 1, 1, 1, 1, 0]
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = True
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True

frames = []
num_frames =kp_data.shape[0]  # Limit for speed
print(f"\nRendering {num_frames} frames with T1R-coxa-aligned model...")

with mujoco.Renderer(mj_model, height=512, width=512) as renderer:
    for t in tqdm(range(num_frames)):
        # Get keypoint T1R coxa position for this frame
        kp_t1r_coxa_pos = kp_data[t, t1r_coxa_idx, :]
        
        # Set root position accounting for the body-to-coxa offset
        # root_pos + t1r_coxa_offset_from_root = kp_t1r_coxa_pos
        # Therefore: root_pos = kp_t1r_coxa_pos - t1r_coxa_offset_from_root
        root_pos = kp_t1r_coxa_pos - t1r_coxa_offset_from_root
        mj_data.qpos[:3] = root_pos
        
        # Forward kinematics to update model
        mujoco.mj_forward(mj_model, mj_data)
        
        # Verify alignment
        if t == 0:
            model_t1r_pos = mj_data.site_xpos[t1r_coxa_site_id]
            alignment_error = jnp.linalg.norm(model_t1r_pos - kp_t1r_coxa_pos)
            print(f"\nFrame 0 alignment check:")
            print(f"  Keypoint T1R coxa: {kp_t1r_coxa_pos}")
            print(f"  Model T1R coxa:    {model_t1r_pos}")
            print(f"  Alignment error:   {alignment_error:.9f}")
            
            if alignment_error < 1e-5:
                print(f"  ✅ Perfect alignment!")
            else:
                print(f"  ⚠ Alignment off by {alignment_error:.6f}")
        
        # Update mocap body positions for keypoint visualization
        for i, mocap_id in mocap_indices.items():
            mj_data.mocap_pos[mocap_id] = kp_data[t, i, :]
            mj_data.mocap_quat[mocap_id] = [1, 0, 0, 0]
        
        # Render
        renderer.update_scene(mj_data, camera=f'track1{suffix}', scene_option=scene_option)
        pixels = renderer.render()
        frames.append(pixels)

# Display video
media.show_video(frames, fps=30)


Keypoint data shape: (607, 50, 3)
Available keypoints: ['Scutellum', 'WingL_base', 'WingR_base', 'Antenna_Base', 'EyeL', 'EyeR', 'WingL_V12', 'WingL_V13', 'WingR_V12', 'WingR_V13']...

✓ Found T1R_ThxCx at index 19
✓ Added 50 mocap bodies with colored sites

✓ Model compiled - Total mocap bodies: 50
✓ Mapped 50 mocap bodies
✓ Found MuJoCo T1R coxa site: tracking[T1R_ThxCx]_fly (ID: 23)

T1R coxa in model reference (with floor offset): [ 0.0317 -0.0209 -0.1522]
Floor offset: -0.125
Root body position in reference: [ 0.     0.    -0.125]
T1R coxa offset from root: [ 0.0317 -0.0209 -0.0272]

Rendering 607 frames with T1R-coxa-aligned model...


  0%|          | 0/607 [00:00<?, ?it/s]


Frame 0 alignment check:
  Keypoint T1R coxa: [0.6286627  0.10494909 0.17492948]
  Model T1R coxa:    [0.62866269 0.1049491  0.17492948]
  Alignment error:   0.000000000
  ✅ Perfect alignment!


This browser does not support the video tag.


# Post Stac Validation

In [16]:
stac_data = ioh5.load(stac_path, enable_jax=True)
print_bout_dict_structure(stac_data, max_depth=3)

BOUT DICTIONARY STRUCTURE
Total keys: 131 (info + 130 bouts)

├── bout_000: <dict: 7 keys>
│   ├── egocentric_site_pos: <jax array: shape=(198, 50, 3), dtype=float32>
│   ├── kp_data: <jax array: shape=(198, 150), dtype=float32>
│   ├── marker_sites: <jax array: shape=(198, 50, 3), dtype=float32>
│   ├── qpos: <jax array: shape=(198, 93), dtype=float32>
│   ├── qvel: <jax array: shape=(198, 92), dtype=float32>
│   ├── xpos: <jax array: shape=(198, 68, 3), dtype=float32>
│   └── xquat: <jax array: shape=(198, 68, 4), dtype=float32>
├── bout_001: <dict: 7 keys>
│   ├── egocentric_site_pos: <jax array: shape=(1355, 50, 3), dtype=float32>
│   ├── kp_data: <jax array: shape=(1355, 150), dtype=float32>
│   ├── marker_sites: <jax array: shape=(1355, 50, 3), dtype=float32>
│   ├── qpos: <jax array: shape=(1355, 93), dtype=float32>
│   ├── qvel: <jax array: shape=(1355, 92), dtype=float32>
│   ├── xpos: <jax array: shape=(1355, 68, 3), dtype=float32>
│   └── xquat: <jax array: shape=(1355, 68, 

In [17]:
def make_vidoes(
    mj_model,
    mj_data,
    qposes_rollout,
    scene_option,
    camera="track1",
    height=512,
    width=512,
):
    """
    Make a video of the rollout and reference superimposed.
    """
    frames = []
    with mujoco.Renderer(mj_model, height=height, width=width) as renderer:
        for t in tqdm(range(len(qposes_rollout))):
            mj_data.qpos = qposes_rollout[t]
            mujoco.mj_forward(mj_model, mj_data)
            
            renderer.update_scene(
                mj_data, camera=f"{camera}", scene_option=scene_option
            )
            renderer.scene.flags[mujoco.mjtRndFlag.mjRND_SHADOW] = False
            pixels = renderer.render()
            frames.append(pixels)
    return frames


In [19]:
# mj_model = env.mj_model
clip_lengths = jnp.asarray(stac_data['info']['clip_lengths'])
spec = mujoco.MjSpec().from_file(flybody_path.as_posix())
floor_spec = mujoco.MjSpec().from_file(floor_path.as_posix())
spawn_frame = floor_spec.worldbody.add_frame(
                pos=[0,0,-.125],
                quat=[1,0,0,0],
            )
spawn_body = spawn_frame.attach_body(spec.body("thorax"), "", suffix='_fly')
mj_model = floor_spec.compile()
mj_data = mujoco.MjData(mj_model)

t = 0
frames=[]
bout = 112 # np.argmax(clip_lengths)
qpos_traj =  stac_data[f'bout_{bout:03d}']['qpos'].copy()
# Set up rendering
height, width = 448, 1936//3
camera = mj_model.camera(0).name
scene_option = mujoco.MjvOption()
scene_option.geomgroup[:] = [1, 1, 1, 0, 0, 0]
scene_option.sitegroup[:] = [1, 1, 1, 1, 1, 0]
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = False

# Render from multiple camera angles
all_frames = []
for cam in [1,2]:
    camera = mj_model.camera(cam).name
    frames = make_vidoes(mj_model, mujoco.MjData(mj_model), qpos_traj, scene_option, camera=camera, height=height, width=width)
    all_frames.append(frames)
all_frames = np.concatenate(all_frames, axis=2)
media.show_video(all_frames, fps=60)


  0%|          | 0/607 [00:00<?, ?it/s]

  0%|          | 0/607 [00:00<?, ?it/s]

This browser does not support the video tag.
